Célula 1 — Markdown de abertura

%md
## Qualidade de Dados

Verificação sistemática das cinco dimensões de qualidade indicadas pelo
enunciado (completude, consistência, unicidade, acurácia, outliers) sobre
as tabelas da camada Gold. Alguns problemas já foram identificados e
tratados durante a Silver (ver documentação daquela camada); esta seção
formaliza a checagem completa e verifica se algo passou despercebido.

**Completude**: verificação de valores nulos em todas as colunas
numéricas da tabela mensal.
[célula: completude]

In [0]:
%sql
SELECT
  COUNT(*) AS total_linhas,
  SUM(CASE WHEN selic_mensal_pct    IS NULL THEN 1 ELSE 0 END) AS nulos_selic,
  SUM(CASE WHEN falencias_requeridas IS NULL THEN 1 ELSE 0 END) AS nulos_fal_req,
  SUM(CASE WHEN falencias_decretadas IS NULL THEN 1 ELSE 0 END) AS nulos_fal_dec,
  SUM(CASE WHEN rj_requeridas        IS NULL THEN 1 ELSE 0 END) AS nulos_rj_req,
  SUM(CASE WHEN rj_deferidas         IS NULL THEN 1 ELSE 0 END) AS nulos_rj_def,
  SUM(CASE WHEN rj_concedidas        IS NULL THEN 1 ELSE 0 END) AS nulos_rj_con
FROM mvp_juros_rj.gold.fato_indicadores_mensais;

**Unicidade**: verificação de meses duplicados.
[célula: unicidade]

In [0]:
%sql
SELECT mes_referencia, COUNT(*) AS qtd
FROM mvp_juros_rj.gold.fato_indicadores_mensais
GROUP BY mes_referencia
HAVING COUNT(*) > 1;

**Consistência**: verificação de faixa de valores (Selic) e de lógica
entre estágios do processo judicial (requerida/deferida/concedida,
requerida/decretada).
[célula: WHERE com as 14 linhas]

In [0]:
%sql
SELECT *
FROM mvp_juros_rj.gold.fato_indicadores_mensais
WHERE selic_mensal_pct < 0 OR selic_mensal_pct > 5
   OR rj_deferidas > rj_requeridas
   OR rj_concedidas > rj_deferidas
   OR falencias_decretadas > falencias_requeridas;

**Outliers**

Identificação de meses em que `rj_requeridas` foge mais de 2 desvios-padrão
da média do período, não indicam erro no dado, mas eventos que merecem
atenção na etapa de Análise.

In [0]:
%sql
WITH stats AS (
  SELECT AVG(rj_requeridas) AS media, STDDEV(rj_requeridas) AS desvio
  FROM mvp_juros_rj.gold.fato_indicadores_mensais
)
SELECT f.mes_referencia, f.rj_requeridas,
       ROUND(stats.media, 1) AS media_geral,
       ROUND(stats.desvio, 1) AS desvio_padrao
FROM mvp_juros_rj.gold.fato_indicadores_mensais f, stats
WHERE f.rj_requeridas > stats.media + 2 * stats.desvio
   OR f.rj_requeridas < stats.media - 2 * stats.desvio;

%md
Os dois outliers identificados fazem sentido à luz do próprio objetivo
do projeto: jan/2024 é o primeiro mês da série (volume ainda baixo, sem
efeito acumulado do ciclo de juros), enquanto jul/2025 coincide com um
dos picos observados na fase mais elevada da Selic dentro da janela
analisada. Ambos serão retomados na seção de Análise.

**Verificações complementares**

Unicidade e consistência de domínio aplicadas também à tabela de
contexto histórico anual (`fato_contexto_anual`).

In [0]:
%sql
-- Unicidade
SELECT ano, COUNT(*) AS qtd FROM mvp_juros_rj.gold.fato_contexto_anual GROUP BY ano HAVING COUNT(*) > 1;

-- Consistência de domínio (tipo_valor só pode ser primario/estimado)
SELECT DISTINCT tipo_valor FROM mvp_juros_rj.gold.fato_contexto_anual;

In [0]:
%sql
SELECT
  SUM(CASE WHEN rj_deferidas > rj_requeridas THEN 1 ELSE 0 END) AS casos_deferida_maior_requerida,
  SUM(CASE WHEN rj_concedidas > rj_deferidas THEN 1 ELSE 0 END) AS casos_concedida_maior_deferida,
  SUM(CASE WHEN falencias_decretadas > falencias_requeridas THEN 1 ELSE 0 END) AS casos_falencia_inconsistente,
  SUM(CASE WHEN selic_mensal_pct < 0 OR selic_mensal_pct > 5 THEN 1 ELSE 0 END) AS casos_selic_fora_faixa
FROM mvp_juros_rj.gold.fato_indicadores_mensais;

In [0]:
%sql
SELECT
  SUM(rj_requeridas)  AS total_requeridas,
  SUM(rj_deferidas)   AS total_deferidas,
  SUM(rj_concedidas)  AS total_concedidas,
  SUM(falencias_requeridas) AS total_fal_requeridas,
  SUM(falencias_decretadas) AS total_fal_decretadas
FROM mvp_juros_rj.gold.fato_indicadores_mensais;

**Achado de qualidade: falso positivo investigado**

A checagem inicial de consistência (comparando mês a mês rj_requeridas,
rj_deferidas e rj_concedidas, e também falencias_requeridas com
falencias_decretadas) sinalizou 14 das 28 linhas como inconsistentes,
porque em alguns meses havia mais casos deferidos ou decretados do que
requeridos naquele mesmo mês.

A investigação mostrou que isso é esperado: os estágios de um processo
judicial (requerimento, deferimento, concessão) não acontecem no mesmo
mês. Um pedido requerido em novembro pode ser deferido só em janeiro,
por exemplo. A comparação correta é olhar o total acumulado do
período, não mês a mês. E isso se confirmou: somando tudo entre
jan/2024 e abr/2026, requeridas (2.236) fica maior ou igual a
deferidas (1.873), que fica maior ou igual a concedidas (680); o mesmo
vale pra falências requeridas (1.760) e decretadas (1.582). Ou seja, a
cadeia de estágios está correta e não existe inconsistência real nos
dados.